In [5]:
import pandas as pd
import json

In [6]:
#load in files
routes = pd.read_csv('data/standard_route_data/routes.txt')
stops = pd.read_csv('data/standard_route_data/stops.txt')
trips = pd.read_csv('data/standard_route_data/trips.txt')
stop_times = pd.read_csv('data/standard_route_data/stop_times.txt')

In [10]:
#stops: parent stations only with lat/lon
stops_parent = stops[stops['location_type'] == 1][['stop_id','stop_name','stop_lat','stop_lon']].copy()

In [11]:
#link stop_times --> trips --> routes ──
merged = stop_times.merge(trips[['trip_id','route_id','service_id']], on='trip_id')

In [14]:
#compute headway per route per stop
def parse_time_minutes(t):
    try:
        parts = str(t).split(':')
        return int(parts[0]) * 60 + int(parts[1])
    except:
        return None
    
merged['arr_min'] = merged['arrival_time'].apply(parse_time_minutes)

In [ ]:
#focus on weekday service only
weekday = merged[merged['service_id'].str.contains('Weekday|weekday|WD', na=False)]
if len(weekday) == 0:
    weekday = merged

In [16]:
#for each route and stop, get sorted arrival times and compute gaps
def compute_headway(group):
    times = group['arr_min'].dropna().sort_values()
    if len(times) < 2:
        return None
    gaps = times.diff().dropna()
    # filter out overnight gaps (>120 min)
    gaps = gaps[gaps < 120]
    return gaps.mean() if len(gaps) > 0 else None

headways = (weekday.groupby(['route_id','stop_id'])
            .apply(compute_headway)
            .reset_index()
            .rename(columns={0:'avg_headway_min'}))

/var/folders/3j/4jrn8x0j7l3_51kmlf95ks9m0000gn/T/ipykernel_25189/2155671598.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_headway)


In [17]:
#avg headway per route across all stops
route_headways = (headways.groupby('route_id')['avg_headway_min']
                  .mean()
                  .reset_index()
                  .rename(columns={'avg_headway_min':'avg_headway_min'}))

In [20]:
#stops per route
stop_to_parent = stops[stops['parent_station'].notna()][['stop_id','parent_station']].copy()

# merge stop_times with parent lookup
st_with_parent = stop_times.merge(stop_to_parent, on='stop_id', how='left')
st_with_parent['parent'] = st_with_parent['parent_station'].fillna(st_with_parent['stop_id'])

# link to route via trips
st_route = st_with_parent.merge(trips[['trip_id','route_id']], on='trip_id')

# unique stops per route
stops_per_route = (st_route.groupby('route_id')['parent']
                   .nunique()
                   .reset_index()
                   .rename(columns={'parent':'num_stops'}))

In [21]:
#build route, stop with lat/lon
route_stop_list = (st_route.groupby('route_id')['parent']
                   .apply(lambda x: list(x.unique()))
                   .reset_index()
                   .rename(columns={'parent':'stop_ids'}))

In [22]:
#merge everything
result = routes[['route_id','route_short_name','route_long_name','route_color','route_text_color']].copy()
result = result.merge(route_headways, on='route_id', how='left')
result = result.merge(stops_per_route, on='route_id', how='left')
result = result.merge(route_stop_list, on='route_id', how='left')

In [24]:
#add stop coordinates
stop_coords = stops_parent.set_index('stop_id')[['stop_lat','stop_lon','stop_name']].to_dict('index')

def get_stop_coords(stop_ids):
    coords = []
    for sid in (stop_ids or []):
        if sid in stop_coords:
            coords.append({
                'id': sid,
                'name': stop_coords[sid]['stop_name'],
                'lat': stop_coords[sid]['stop_lat'],
                'lon': stop_coords[sid]['stop_lon']
            })
    return coords

result['stop_coords'] = result['stop_ids'].apply(get_stop_coords)

In [25]:
#export to json
# Only subway routes (route_type == 1)
subway_routes = routes[routes['route_type'] == 1]['route_id'].tolist()
result_subway = result[result['route_id'].isin(subway_routes)].copy()

# Round headway
result_subway['avg_headway_min'] = result_subway['avg_headway_min'].round(1)

output = []
for _, row in result_subway.iterrows():
    output.append({
        'route_id': row['route_id'],
        'name': row['route_short_name'],
        'long_name': row['route_long_name'],
        'color': '#' + str(row['route_color']),
        'text_color': '#' + str(row['route_text_color']),
        'avg_headway_min': row['avg_headway_min'],
        'num_stops': int(row['num_stops']) if pd.notna(row['num_stops']) else 0,
        'stops': row['stop_coords']
    })

In [26]:
with open('data/subway_routes_processed.json', 'w') as f:
    json.dump(output, f)

print(f"Exported {len(output)} subway routes")
print("\nSample headways:")
print(result_subway[['route_id','avg_headway_min','num_stops']].sort_values('avg_headway_min').to_string())

Exported 28 subway routes

Sample headways:
   route_id  avg_headway_min  num_stops
16       GS              4.3          2
11        L              5.3         24
26        7              5.4         22
27       7X              5.8         18
19        1              6.3         38
24        6              7.0         38
25       6X              7.5         29
21        3              8.3         34
7         M              8.5         36
14        R              8.6         52
1         C              9.1         40
5         F              9.2         59
3         B              9.3         37
8         G              9.6         21
10        Z             10.1         21
9         J             10.3         30
13        Q             10.4         34
4         D             10.5         41
2         E             11.5         36
17       FS             12.0          4
22        4             12.5         54
20        2             12.7         71
0         A             13.1        

In [27]:
import numpy as np
np.set_printoptions(threshold=np.inf)
print(result_subway[['route_id','avg_headway_min','num_stops']].sort_values('avg_headway_min').to_string())

   route_id  avg_headway_min  num_stops
16       GS              4.3          2
11        L              5.3         24
26        7              5.4         22
27       7X              5.8         18
19        1              6.3         38
24        6              7.0         38
25       6X              7.5         29
21        3              8.3         34
7         M              8.5         36
14        R              8.6         52
1         C              9.1         40
5         F              9.2         59
3         B              9.3         37
8         G              9.6         21
10        Z             10.1         21
9         J             10.3         30
13        Q             10.4         34
4         D             10.5         41
2         E             11.5         36
17       FS             12.0          4
22        4             12.5         54
20        2             12.7         71
0         A             13.1         66
23        5             14.6         55
